In [1]:
import mlflow
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification

In [2]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from loguru import logger

import pandas as pd
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

import shap
import matplotlib.pyplot as plt
from typing import Dict

In [3]:
mlflow.set_tracking_uri("http://localhost:5000")

In [4]:
def generate_synthetic_data():
    """
    Generate synthetic data for classification
    """
    X, y = make_classification(
        n_samples=5000,
        n_features=20,
        n_informative=15,
        n_redundant=3,
        n_classes=2,
        random_state=42
    )

    df = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(X.shape[1])])
    df['feature_interaction'] = df['feature_0'] * df['feature_1']
    df['feature_sum'] = df['feature_2'] + df['feature_3']
    return df, y

In [5]:
df, y = generate_synthetic_data()
X = df.values  # Convert DataFrame to numpy array

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Define hyperparameter grid for tuning
param_grid: Dict[str, list] = {
    'n_estimators': [50, 100],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 5],
    'subsample': [0.8, 1.0]
}

# Perform hyperparameter tuning using GridSearchCV
gb = GradientBoostingClassifier()
grid_search = GridSearchCV(gb, param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Retrieve the best model from the grid search
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

In [8]:
def calculate_metrics(y_true, y_pred, y_proba):
    """
    Calculate classification metrics
    """
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_proba)
    }
    return metrics

metrics = calculate_metrics(y_test, y_pred, y_proba)

In [10]:
with mlflow.start_run():
    mlflow.log_params(grid_search.best_params_)
    mlflow.log_metrics(metrics)
    mlflow.set_tag("model_description", "Gradient Boosting Classifier trained with GridSearchCV for hyperparameter tuning.")

    # Log the trained model with inferred input signature
    signature = infer_signature(X_train, best_model.predict(X_train))
    mlflow.sklearn.log_model(best_model, "gradient_boosting_model", signature=signature)

    # SHAP feature importance analysis
    explainer = shap.Explainer(best_model, X_train)
    shap_values = explainer(X_test[:100])

    # Generate and log SHAP summary plot
    shap_plot_path = "shap_summary_plot.png"
    plt.figure()
    shap.summary_plot(shap_values, X_test[:100], show=False)
    plt.savefig(shap_plot_path, bbox_inches="tight")
    plt.close()
    mlflow.log_artifact(shap_plot_path)

logger.info("Best Parameters: {}", grid_search.best_params_)
logger.info("Evaluation Metrics: {}", metrics)

🏃 View run intrigued-cod-815 at: http://localhost:5000/#/experiments/0/runs/104e7d607f44454fbba50d2a0aec72e4
🧪 View experiment at: http://localhost:5000/#/experiments/0


MlflowException: API request to http://localhost:5000/api/2.0/mlflow-artifacts/artifacts/0/104e7d607f44454fbba50d2a0aec72e4/artifacts/gradient_boosting_model/python_env.yaml failed with exception HTTPConnectionPool(host='localhost', port=5000): Max retries exceeded with url: /api/2.0/mlflow-artifacts/artifacts/0/104e7d607f44454fbba50d2a0aec72e4/artifacts/gradient_boosting_model/python_env.yaml (Caused by ResponseError('too many 500 error responses'))